# CHRONO | The Trident Signal System
### Powered by Gemma 4 

**Mission:** Detect pre-cancerous metabolic fingerprints from routine blood test history years before structural changes occur.

This notebook demonstrates the full CHRONO pipeline:
1. **Ingestion:** Gemma 4 Vision extracts markers from lab report images.
2. **Engine:** Computing WIV, BAV, and ICV velocity signals.
3. **Triage:** Protocol-99 ReAct agent investigating the anomaly.

## 1. Setup and Dependencies

In [ ]:
!pip install -q transformers accelerate pillow torch

import torch
from PIL import Image
from transformers import AutoProcessor, AutoModelForImageTextToText
import json
import os

## 2. Layer 1: Document Ingestion (Gemma 4 Vision)
We use **Gemma 4 E4B** to read a photographed lab report and extract structured JSON data.

In [ ]:
model_id = "google/gemma-3-4b-it"
processor = AutoProcessor.from_pretrained(model_id)
model = AutoModelForImageTextToText.from_pretrained(
    model_id, 
    torch_dtype=torch.bfloat16, 
    device_map="auto"
)

def extract_markers(image_path):
    prompt = "Extract ALL blood test values from this image. Return ONLY valid JSON with keys: test_date, markers (name, value, unit)."
    image = Image.open(image_path).convert("RGB")
    
    inputs = processor(text=prompt, images=image, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=1024)
    
    return processor.decode(outputs[0], skip_special_tokens=True)

## 3. Layer 3: Trident Signal Engine
We compute the velocity of the three core signals against the patient's personal baseline.

In [ ]:
from engine.personal_baseline import PersonalBaseline
from engine.mcf_scorer import MCFScorer
from engine.wiv_calculator import WIVCalculator

# Mock data for demonstration
history = [
    ("2022-01-01", {"ldh": 165, "rdw": 12.8, "glucose": 92}),
    ("2023-01-01", {"ldh": 185, "rdw": 13.5, "glucose": 97}),
    ("2024-01-01", {"ldh": 214, "rdw": 14.2, "glucose": 101})
]

baseline = PersonalBaseline("priya_001")
for dt, vals in history: 
    baseline.add_test(dt, vals)

calc = WIVCalculator(baseline)
result = calc.compute(history)
print(f"Warburg Index Velocity: {result.wiv:+.4f} units/month")

## 4. Layer 4: Protocol-99 Agent (Gemma 4 26B)
When MCF > 0.61, the ReAct agent investigates.

In [ ]:
from agent.protocol99_react import Protocol99Agent

# Simulate the agentic reasoning trace
agent = Protocol99Agent("priya_001", baseline, result) # result is mock result here
trace = agent.run_simulation()

for step in trace:
    print(f"[{step['role'].upper()}] {step['content']}")